In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load BIOSCAN and CAOS metadata

In [ ]:
metadata = pd.read_csv("data/metadata.tsv", sep="\t")

/var/folders/k4/f_bpyr193w1dyd_9bdnnr3pc0000gn/T/ipykernel_40333/1695267852.py:2: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv("data/metadata.tsv", sep="\t")


In [ ]:
bioscan = pd.read_csv("data/BIOSCAN_5M_Insect_Dataset_metadata.tsv", sep="\t")

# Filter Keyence and non-Keyence samples

In [22]:
non_keyence = metadata[
    (metadata['photographer'] != 'CBG Robotic Imager') &
    (metadata['meta'] != 'Microplate')
]

In [ ]:
# Identify samples in BIOSCAN-5M that are not from Keyence microscopes
non_keyence_bioscan = bioscan[
    bioscan['processid'].isin(non_keyence['processid'])
]
# INFO: 2 samples from the BIOSCAN dataset where not in the BOLD metadata but I checked them and they are Keyence images
keyence_bioscan = bioscan.drop(non_keyence_bioscan.index) 

In [24]:
non_keyence_bioscan

,processid,sampleid,taxon,phylum,class,order,family,subfamily,genus,species,...,province_state,coord-lat,coord-lon,image_measurement_value,area_fraction,scale_factor,inferred_ranks,split,index_bioscan_1M_insect,chunk
260,GMCED7816-21,BIOUG71920-F04,Sciaridae,Arthropoda,Insecta,Diptera,Sciaridae,NaN,NaN,NaN,...,Guanacaste,10.92900,-85.46400,NaN,0.1783,1.2891,0,pretrain,937487.0,c8
399,YDBB7781-21,BIOUG75301-E01,Proteinus parvulus,Arthropoda,Insecta,Coleoptera,Staphylinidae,Proteininae,Proteinus,Proteinus parvulus,...,Yukon Territory,60.59400,-134.95000,NaN,0.3453,1.4414,0,test,179861.0,NaN
2316,YGEN730-22,BIOUG75697-A07,Simulium maculatum,Arthropoda,Insecta,Diptera,Simuliidae,Simuliinae,Simulium,Simulium maculatum,...,Yukon Territory,60.92500,-136.76700,NaN,0.2009,1.4727,0,test,NaN,NaN
3437,CRALC15745-21,BIOUG72854-A08,Allodorus Malaise3443,Arthropoda,Insecta,Hymenoptera,Braconidae,Brachistinae,Allodorus,Allodorus Malaise3443,...,Puntarenas,9.01500,-83.00400,NaN,0.4942,2.1055,0,other_heldout,927366.0,6
11833,CRALC27767-21,BIOUG72732-D04,Nealiolus Malaise3681,Arthropoda,Insecta,Hymenoptera,Braconidae,Brachistinae,Nealiolus,Nealiolus Malaise3681,...,Puntarenas,9.01500,-83.00400,NaN,0.7344,23.1836,0,other_heldout,936837.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5148830,PLQAE443-20,BIOUG53118-E05,encyrMalaise01 Malaise8695,Arthropoda,Insecta,Hymenoptera,Encyrtidae,unassigned Encyrtidae,encyrMalaise01,encyrMalaise01 Malaise8695,...,Guanacaste,10.76000,-85.33400,4100.0,0.6179,2.5547,0,other_heldout,NaN,3
5150054,PLPCC1367-20,BIOUG61825-A07,encyrMalaise01 Malaise7116,Arthropoda,Insecta,Hymenoptera,Encyrtidae,unassigned Encyrtidae,encyrMalaise01,encyrMalaise01 Malaise7116,...,Guanacaste,10.76300,-85.33400,NaN,0.4956,2.0547,0,other_heldout,NaN,a
5150477,GMSJC874-18,BIOUG38123-A01,Balclutha frontalis,Arthropoda,Insecta,Hemiptera,Cicadellidae,Deltocephalinae,Balclutha,Balclutha frontalis,...,Gauteng,-26.18257,27.99755,191.0,0.0345,0.4609,0,train,NaN,c
5150730,GMSJD597-18,BIOUG38134-F03,Phoridae,Arthropoda,Insecta,Diptera,Phoridae,NaN,NaN,NaN,...,Gauteng,-26.18257,27.99755,26697.0,0.0340,0.4688,0,pretrain,NaN,0d


In [26]:
# Save processids of non-Keyence BIOSCAN samples
non_keyence_bioscan['processid'].to_csv("data/nonKeyence_processids.txt", index=False, header=False)

In [ ]:
for index, row in non_keyence_bioscan.iterrows():
    src_path = os.path.join("data/BIOSCAN_5M_images/", f"{row['processid']}.jpg")
    dest_path = os.path.join("data/BIOSCAN_5M_nonKeyence_images/", f"{row['processid']}.jpg")
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    os.rename(src_path, dest_path)

In [ ]:
import sys
from ..src.scaledetection import main as scale_detection_main

_prev_argv = sys.argv
sys.argv = [
	'scaledetection',
	'--model', 'models/yolov8m_train/weights/best.pt',
	'--image_dir', 'data/BIOSCAN_5M_nonKeyence_images/',
	'--output_dir', 'results/BIOSCAN_5M_nonKeyence/'
]
try:
	scale_results = scale_detection_main()
finally:
	sys.argv = _prev_argv

scale_results